In [13]:
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.types import Command,interrupt
from langchain_core.messages import AIMessage,HumanMessage,ToolMessage
from typing import Annotated,TypedDict
import os
import sqlite3
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

True

In [14]:
class ApprovalState(TypedDict):
    task:str
    plan:str
    result:str
    approved:bool

def plan_task(state:ApprovalState)->dict:
    plan=f"I will execute the task {state['task']} using the following steps: 1.setup 2.execute 3.verify"
    print(f"Generated Plan: {plan}")

    approval=interrupt(
    {
        "question":"Do you approve this plan ?",
        "plan": plan,
        "options":["yes","no","verify"]
    })
    return {"plan":plan, "approved":approval=="yes"}

def execute_task(state:ApprovalState)->dict:
    if not state['approved']:
        return {"msg":"Cancelled by the USER"}
    result=f"Successfully executed {state['task']}"

    return {"result":result}

db_path="Memory/HTL.db"
conn=sqlite3.connect(db_path,check_same_thread=False)
memory=SqliteSaver(conn)

graph=StateGraph(ApprovalState)

graph.add_node("plan",plan_task)
graph.add_node("execute",execute_task)
graph.add_edge(START,"plan")
graph.add_edge("plan","execute")
graph.add_edge("execute",END)

app=graph.compile(checkpointer=memory)


In [15]:
config={"configurable":{"thread_id":"my-user123"}}
result=app.invoke({
    "task":"Migrate Database","plan":"","result":"","approved":"False"
},config)

print(f"Result is {result}")
state=app.get_state(config)
print(f"State is : {state.values['task']}")

final_result=app.invoke((Command(resume="yes")),config)
print(f"Final-Result:{final_result}")


Generated Plan: I will execute the task Migrate Database using the following steps: 1.setup 2.execute 3.verify
Result is {'task': 'Migrate Database', 'plan': '', 'result': '', 'approved': 'False', '__interrupt__': [Interrupt(value={'question': 'Do you approve this plan ?', 'plan': 'I will execute the task Migrate Database using the following steps: 1.setup 2.execute 3.verify', 'options': ['yes', 'no', 'verify']}, id='53550b6494bd2bd5db737481e843951d', response_schema=None)]}
State is : Migrate Database
Generated Plan: I will execute the task Migrate Database using the following steps: 1.setup 2.execute 3.verify
Final-Result:{'task': 'Migrate Database', 'plan': 'I will execute the task Migrate Database using the following steps: 1.setup 2.execute 3.verify', 'result': 'Successfully executed Migrate Database', 'approved': True}
